# LitaLite v6 + depth — shrub only (crash fix) — Kaggle
The earlier v6 run crashed on **shrub** at iteration 5001 (start of phase 2: 3D illumination smoothness + white-patch on reflectance) with a CUDA *index out of bounds*.
Fix (commit after `07fb904`): the phase-2 losses now use only Gaussians with finite position/illumination, centre positions before the distance matrix, replace non-finite distances before `topk`, clamp neighbour indices, and check the visibility mask.
Diagnostics: at each relight update the log prints the counts of bad Gaussians (`bad_xyz / bad_illum / bad_sh`); for the first 20 iterations of phase 2 (`--debug_sync 20`) CUDA is synchronised after every loss term, so a remaining error is pinned to its term.
Run: **shrub**, v6 + depth (weight 0.05), **official test views** 1, 17, 24, 31, 7. Compare: shrub v4 15.19 / 0.641 / 0.221.
Input: dataset **lom-full** (Add Input!). GPU T4, Internet on. **Save Version -> Save & Run All** (~1 h). Results in `shrub_v6_results.txt`.


In [ ]:
import os, json, shutil, glob, torch
from pathlib import Path
assert torch.cuda.is_available(), "Enable a GPU in Settings"
g = torch.cuda.get_device_properties(0); print(g.name, torch.__version__)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{g.major}.{g.minor}"; os.environ["MAX_JOBS"] = "2"
TMP = Path("/kaggle/temp"); TMP.mkdir(exist_ok=True); WORK = Path("/kaggle/working")
SCENES = ["shrub"]
def find_scene(name):
    for t in Path("/kaggle/input").rglob("transforms_test.json"):
        d = t.parent
        if d.name == name and (d/"low").is_dir() and (d/"high").is_dir() and list(d.glob("colmap_sparse/*/cameras.bin")):
            return d
    return None
FOUND = {s: find_scene(s) for s in SCENES}
for s, d in FOUND.items(): print(f"{s:6s}", d)
assert any(FOUND.values()), "no LOM scene found in /kaggle/input"
def prep(name):
    d = FOUND[name]; out = TMP/"lom"/name; shutil.rmtree(out, ignore_errors=True)
    shutil.copytree(d/"low", out/"images"); shutil.copytree(d/"high", out/"gt")
    sp = sorted(d.glob("colmap_sparse/*/cameras.bin"))[0].parent
    shutil.copytree(sp, out/"sparse"/"0")
    test = [os.path.splitext(os.path.basename(f["file_path"]))[0] for f in json.load(open(d/"transforms_test.json"))["frames"]]
    print(f"[{name}] {len(os.listdir(out/'images'))} dark, {len(os.listdir(out/'gt'))} references, official test views {test}")
    return out, test

In [ ]:
# ---- clone halo-gs (needs the shrub-fix commit pushed) + vanilla gaussian-splatting, build (~10 min) ----
%cd /kaggle/temp
!rm -rf halo-gs gaussian-splatting
!git clone -q https://github.com/arpitpaliwal007/halo-gs
!cd halo-gs && git log --oneline -1 && grep -q debug_sync train_glow.py && echo "shrub-fix code OK" || echo "!!! FIX NOT PUSHED"
!git clone -q --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /kaggle/temp/gaussian-splatting
!grep -q cfloat submodules/simple-knn/simple_knn.cu || sed -i '1i #include <cfloat>' submodules/simple-knn/simple_knn.cu
!grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '1i #include <cstdint>' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
from pathlib import Path
f = Path("scene/dataset_readers.py"); t = f.read_text()
key = 'assert False, "Colmap camera model not handled'
if key in t:   # any COLMAP model (buu is not PINHOLE): use the focal length(s), ignore distortion, as in all earlier runs
    i = t.index(key); ls = t.rfind("\n", 0, i); le = t.index("\n", i)
    ind = t[ls+1:i]
    fb = (f"{ind}focal_length_x = intr.params[0]\n"
          f"{ind}focal_length_y = intr.params[1] if intr.model in ('PINHOLE','OPENCV','OPENCV_FISHEYE','FULL_OPENCV','FOV','THIN_PRISM_FISHEYE') else intr.params[0]\n"
          f"{ind}FovY = focal2fov(focal_length_y, height)\n{ind}FovX = focal2fov(focal_length_x, width)")
    t = t[:ls+1] + fb + t[le:]; f.write_text(t); print("patched: all camera models accepted")
assert key not in f.read_text()
!pip -q install plyfile lpips scikit-image ninja
!pip -q install --no-build-isolation ./submodules/diff-gaussian-rasterization ./submodules/simple-knn
!python -c "import diff_gaussian_rasterization, simple_knn; print('rasteriser built')"

In [ ]:
# ---- check the depth network loads (fails here, before any training, if it cannot) ----
import subprocess
subprocess.run("pip -q install 'transformers>=4.45'", shell=True)
from transformers import pipeline
import numpy as np
from PIL import Image
_p = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
_d = _p(Image.fromarray((np.random.rand(375, 500, 3) * 255).astype("uint8")))["predicted_depth"]
print("Depth Anything V2 Small OK, output shape", tuple(np.asarray(_d).shape)); del _p

In [ ]:
# ---- runs ----
import time, subprocess, numpy as np, lpips
from PIL import Image, ImageDraw
from skimage.metrics import peak_signal_noise_ratio as P, structural_similarity as S
GS = "/kaggle/temp/gaussian-splatting"; HG = "/kaggle/temp/halo-gs"
V6 = ("--mode litalite --iterations 30000 --save_iterations 30000 --test_every 5000 --quiet "
      "--illum_channels 3 --denoise_target n2n_unet --target_curve nova --nova_mean 0.5 "
      "--nova_mean_rule whitepatch_exposure --relight3d --w_anchor 0.1 --dump_targets 3").split()
DEPTH = ["--w_depth", "0.05"]
RUNS = [  # name, scene, test views (None = official), extra flags
    ("shrub_v6_depth", "shrub", None, DEPTH + ["--debug_sync", "20"]),
]
lp = lpips.LPIPS(net="alex", verbose=False).cuda()
L = lambda p: np.asarray(Image.open(p).convert("RGB"), np.float32) / 255
T = lambda a: torch.from_numpy(a).permute(2, 0, 1)[None].cuda() * 2 - 1
rows = {}
for name, scene, views, extra in RUNS:
    try:
        data, official = prep(scene); test = views or official; out = f"/kaggle/temp/out/{name}"
        cmd = ["python", "train_glow.py", "-s", str(data), "-m", out, "--gs_root", GS, *V6, *extra,
               "--split", "views", "--test_views", *test, "--test_views_by", "name",
               "--clean_gt_dir", str(data/"gt"), "--tag", name]
        t0 = time.time()
        with open(WORK/f"{name}_train_log.txt", "w") as lf:
            r = subprocess.run(cmd, cwd=HG, stdout=lf, stderr=subprocess.STDOUT)
        mins = (time.time() - t0) / 60
        for l in open(WORK/f"{name}_train_log.txt"):
            if any(k in l for k in ("white-patch exposure", "monocular depth", "NOVA-GS target", "vs clean reference", "[relight3d]")): print(f"[{name}]", l.strip()[:150])
        if r.returncode != 0:   # show where it stopped
            print(f"----- last 40 lines of {name}_train_log.txt -----"); print("".join(open(WORK/f"{name}_train_log.txt").readlines()[-40:]))
        assert r.returncode == 0 and os.path.exists(f"{out}/point_cloud/iteration_30000/point_cloud.ply"), f"training failed, see {name}_train_log.txt"
        subprocess.run(["python", "render_glow.py", "-m", out, "--gs_root", GS, "--iteration", "30000", "--skip_train", "--quiet"],
                       cwd=HG, check=True, stdout=subprocess.DEVNULL)
        bdir = (glob.glob(f"{out}/renders/iteration_30000/test/bright") or glob.glob(f"{out}/**/test/bright", recursive=True))[0]
        m, W, H, pad = [], 300, 225, 4
        sheet = Image.new("RGB", (3*(W+pad)+pad, 24+len(test)*(H+pad)+pad), (18, 18, 20)); d = ImageDraw.Draw(sheet)
        for j, t in enumerate(["dark", name, "reference"]): d.text((pad+j*(W+pad)+4, 6), t, fill=(235, 235, 235))
        for i, v in enumerate(test):
            pf = [p for p in glob.glob(f"{bdir}/*") if os.path.basename(p).split(".")[0] == v][0]
            shutil.copy(pf, WORK/f"render_{name}_view{v}.png")
            ref = L(glob.glob(f"{data}/gt/{v}.*")[0]); pred = L(pf)
            if pred.shape != ref.shape: pred = np.asarray(Image.fromarray((pred*255).astype(np.uint8)).resize(ref.shape[1::-1]), np.float32)/255
            with torch.no_grad(): l = lp(T(pred), T(ref)).item()
            m.append((P(ref, pred, data_range=1), S(ref, pred, channel_axis=2, data_range=1), l))
            for j, im in enumerate([L(glob.glob(f"{data}/images/{v}.*")[0]), pred, ref]):
                sheet.paste(Image.fromarray((im*255).astype(np.uint8)).resize((W, H)), (pad+j*(W+pad), 24+i*(H+pad)))
        sheet.save(WORK/f"{name}_vs_ref.png")
        if os.path.isdir(f"{out}/targets"):
            for f in sorted(os.listdir(f"{out}/targets")): shutil.copy(f"{out}/targets/{f}", WORK/f"{name}_targets_{f}")
        mm = np.mean(m, 0); rows[name] = (*mm, mins)
        print(f"{name:16s} PSNR {mm[0]:.2f}  SSIM {mm[1]:.3f}  LPIPS {mm[2]:.3f}  train {mins:.1f} min | per view " +
              ", ".join(f"{v}: {x[0]:.2f}" for v, x in zip(test, m)))
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"[{name}] FAILED: {e}")
    tab = "run               PSNR    SSIM   LPIPS  train(min)\n" + "".join(f"{k:16s}  {v[0]:6.2f}  {v[1]:.3f}  {v[2]:.3f}  {v[3]:6.1f}\n" for k, v in rows.items())
    tab += "compare: shrub v4 (official) 15.19/0.641/0.221 | first v6 shrub run: 19.36 at 5k, then crashed\n"
    open(WORK/"shrub_v6_results.txt", "w").write(tab)
print(tab)